# PARC2026 — G1 Generalization / Safe Augmentation Screening

M3でshortlistされた最大2モデルについて、normal best recipeを基準に
safe augmentation / targeted public supplementalを小規模比較します。

候補:
- baseline best recipe
- brightness / contrast
- mild color jitter
- mild crop-resize
- targeted public supplemental（provenance必須）

Gate:
- Track2改善が主目的
- Track1 regressionを必ず確認
- M3 shortlistが無い間はtraining/evalを開始しない
- supplemental dataはdataset id/revision/license/episode manifestが無いと使用不可


In [ ]:
# 0/3 Drive + M3 shortlist gate
import json
from pathlib import Path
from google.colab import drive
drive.mount("/content/drive")
DRIVE=Path("/content/drive/MyDrive/parc2026-cache")
M3=DRIVE/"model-benchmark-v1/model_shortlist.json"
if not M3.exists(): raise RuntimeError("M3 model_shortlist.json is required before G1")
shortlist=json.loads(M3.read_text()); assert shortlist.get("status")=="DECIDED"
models=shortlist.get("models",[]); assert 1 <= len(models) <= 2
print("shortlist:",models)
OUT=DRIVE/"generalization-screening-v1"; OUT.mkdir(parents=True,exist_ok=True)


In [ ]:
# 1/3 Pre-register candidates and regression contract
import json
CANDIDATES=[{"id":"baseline_best_recipe","type":"baseline","params":{}},{"id":"brightness","type":"visual_aug","params":{"strength":"mild"}},{"id":"contrast","type":"visual_aug","params":{"strength":"mild"}},{"id":"mild_color_jitter","type":"visual_aug","params":{"strength":"mild"}},{"id":"mild_crop_resize","type":"visual_aug","params":{"strength":"mild"}},{"id":"targeted_public_supplemental","type":"supplemental","params":{"requires_provenance":true}}]
contract={"schema_version":1,"models":models,"candidates":CANDIDATES,"same_normal_dataset_recipe":True,"same_eval_seed_set":True,"primary_target":"track2_success_rate","regression_gate":"track1_success_rate","training_loss_alone_is_not_sufficient":True}
(OUT/"generalization_contract.json").write_text(json.dumps(contract,indent=2)+"\n"); print(json.dumps(contract,indent=2))


In [ ]:
# 2/3 Result gate — fill only after comparable runs exist
import json
results_path=OUT/"candidate_results.json"
if not results_path.exists():
    print("G1 BLOCKED: candidate_results.json is not present yet."); print("Run the shortlist models under the frozen M3 budget/seed contract, then place results here.")
else:
    data=json.loads(results_path.read_text()); rows=data.get("results",[]); assert rows, "empty results"
    for r in rows:
        for k in ["model","candidate","track1_success_rate","track2_success_rate"]: assert k in r,(k,r)
    baseline={r["model"]:r for r in rows if r["candidate"]=="baseline_best_recipe"}; accepted=[]
    for r in rows:
        b=baseline.get(r["model"])
        if not b: continue
        no_t1_regression=r["track1_success_rate"] >= b["track1_success_rate"]; t2_gain=r["track2_success_rate"] > b["track2_success_rate"]
        if no_t1_regression and (r["candidate"]=="baseline_best_recipe" or t2_gain): accepted.append(r)
    if not accepted: raise RuntimeError("No augmentation/supplemental candidate passes Track1 regression gate.")
    winner=max(accepted,key=lambda r:(r["track2_success_rate"],r["track1_success_rate"]))
    result={"status":"PASS","selected_model":winner["model"],"selected_augmentation":winner["candidate"],"track1_success_rate":winner["track1_success_rate"],"track2_success_rate":winner["track2_success_rate"]}
    (OUT/"generalization_result.json").write_text(json.dumps(result,indent=2)+"\n"); print(json.dumps(result,indent=2)); print("=== G1: PASS ===")
